In [2]:
import kagglehub
import pandas as pd
import os
import ast
import html
import re
import string
import unicodedata
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk 
nltk.download('punkt_tab')
nltk.download('stopwords')

/Users/jelenalazovic/ml/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/jelenalazovic/ml/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Error loading punkt_tab: <urlopen error [Errno 8] nodename
[nltk_data]     nor servname provided, or not known>
[nltk_data] Error loading stopwords: <urlopen error [Errno 8] nodename
[nltk_data]     nor servname provided, or not known>


False

Loading raw data

In [4]:
path = kagglehub.dataset_download("shuyangli94/foodcom-recipes-with-search-terms-and-tags")
csv_path = os.path.join(path, 'recipes_w_search_terms.csv')
df = pd.read_csv(csv_path)

def get_cuisine(x):
    if not isinstance(x, str):
        return None
    x_lower = x.lower()
    if 'italian' in x_lower:
        return 'Italian'
    if 'indian' in x_lower:
        return 'Indian'
    return None

italian_indian_df = df[df['search_terms'].apply(lambda x: isinstance(x, str) and ('italian' in x.lower() or 'indian' in x.lower()))].copy()
italian_indian_df['cuisine'] = italian_indian_df['search_terms'].apply(get_cuisine)
final_df = italian_indian_df[['name', 'steps', 'cuisine']].reset_index(drop=True)
final_df['steps'] = final_df['steps'].apply(lambda x: ' '.join(ast.literal_eval(x)) if isinstance(x, str) else x)
final_df['name'] = final_df['name'].str.lower()
final_df['steps'] = final_df['steps'].str.lower()

indian_all = final_df[final_df['cuisine'] == 'Indian']
italian_sample = final_df[final_df['cuisine'] == 'Italian'].sample(n=len(indian_all), random_state=42)

final_df = pd.concat([italian_sample, indian_all], ignore_index=True)
final_df.to_csv('./data/recipes_raw.csv')

In [5]:
non_food_signals = ['scalp', 'shampoo', 'massage your']
final_df['suspicious'] = final_df['steps'].str.lower().apply(
    lambda x: any(w in x for w in non_food_signals))
print(final_df[final_df['suspicious']][['name', 'cuisine']])

final_df = final_df[~final_df['suspicious']].drop(columns='suspicious').reset_index(drop=True)

                                                    name cuisine
10911  magical transformation from very rough and dry...  Indian
12027             homemade scrub to get rid of dead skin  Indian
13107                   silky hair with an egg treatment  Indian


Basic checks

In [6]:
final_df["name"] = (
    final_df["name"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
final_df["steps"] = (
    final_df["steps"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print('Number of null rows\n', final_df.isna().sum())
print('Number of duplicated rows', final_df.duplicated(subset='steps').sum())
df_no_dub = final_df.drop_duplicates(subset='steps').reset_index(drop=True)

Number of null rows
 name       0
steps      0
cuisine    0
dtype: int64
Number of duplicated rows 14


Text cleaning (HTML entities, invisible/control characters)

In [7]:
# zero-width space (200b), line/paragraph separator (2028/2029), BOM (feff), nbsp (a0)
_invisible = [0x200b, 0x2028, 0x2029, 0xfeff, 0xa0]
INVISIBLE_CHARS = re.compile('[' + ''.join(chr(c) for c in _invisible) + ']')
CONTROL_CHARS = re.compile('[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]')

def clean_text(text):
    if not isinstance(text, str):
        return text
    text = html.unescape(text)
    text = unicodedata.normalize('NFKC', text)
    text = INVISIBLE_CHARS.sub(' ', text)
    text = CONTROL_CHARS.sub(' ', text)
    text = re.sub(r'-{2,}', '-', text)  # collapse repeated hyphens (e.g. "----") into one
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_clean = df_no_dub.copy()
df_clean['name'] = df_clean['name'].apply(clean_text)
df_clean['steps'] = df_clean['steps'].apply(clean_text)

In [8]:
from nltk.tokenize import sent_tokenize

noise_signals = ['read more', 'hit play', 'www']

def remove_noisy_sentences(text):
    sentences = sent_tokenize(text)
    kept = [s for s in sentences if not any(w in s.lower() for w in noise_signals)]
    return ' '.join(kept)

df_clean['steps'] = df_clean['steps'].apply(remove_noisy_sentences)

Tokenization

In [9]:
df_clean['steps_tokens'] = df_clean['steps'].apply(word_tokenize)
df_clean.to_csv('./data/recipes_tokenized.csv')

In [10]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [t for t in tokens if t not in stop_words]

def remove_punctuation(tokens):
    return [token for token in tokens if token not in string.punctuation]

# steps_tokens keeps stopwords here - it's the seq2seq target; stopwords are removed
# later from a copy (steps_tokens_bow) for the BoW classifier. Punctuation carries no
# useful signal either way, so it's dropped from steps_tokens too.
df_clean['steps_tokens'] = df_clean['steps_tokens'].apply(remove_punctuation)

Statistical analysis

In [11]:
df_clean['number_of_tokens'] = df_clean['steps_tokens'].apply(len)
print(df_clean['number_of_tokens'].describe())

max_len = int(df_clean['number_of_tokens'].quantile(0.95))
df_clean_no_outliers = df_clean[
    df_clean['number_of_tokens'].between(13, max_len - 1)
].copy()
print(df_clean_no_outliers['cuisine'].value_counts())

count    13095.000000
mean       123.042688
std         84.319923
min          1.000000
25%         68.000000
50%        104.000000
75%        155.000000
max       1220.000000
Name: number_of_tokens, dtype: float64
cuisine
Indian     6221
Italian    6118
Name: count, dtype: int64


Additional steps_tokens cleaning (numbers, units, artifacts, fractions, hyphens)

In [12]:
MEASUREMENT_WORDS = {
    "cup", "cups",
    "tbsp", "tablespoon", "tablespoons",
    "tsp", "teaspoon", "teaspoons",
    "oz", "ounce", "ounces",
    "lb", "lbs", "pound", "pounds",
    "qt", "quart", "quarts",
    "pint", "pints",
    "g", "kg", "mg",
    "ml", "l",
    "inch", "inches",
    "degree", "degrees",
    "minute", "minutes",
    "hour", "hours"
}

def remove_numbers_measurements(tokens):
    cleaned = []
    for token in tokens:
        token = token.lower()
        # remove pure numbers and fractions
        if re.fullmatch(r"[\d¼½¾⁄/.-]+", token):
            continue
        if token in MEASUREMENT_WORDS:
            continue
        cleaned.append(token)
    return cleaned

ARTIFACTS = {
    "'s", "'re", "'ve", "'ll", "'d", "'m", "n't", "--"
}

def remove_artifacts(tokens):
    return [t for t in tokens if t not in ARTIFACTS]

def normalize_unicode_fractions(tokens):
    replacements = {"½": "1/2", "¼": "1/4", "¾": "3/4", "⁄": "/"}
    normalized = []
    for token in tokens:
        for old, new in replacements.items():
            token = token.replace(old, new)
        normalized.append(token)
    return normalized

def split_hyphenated(tokens):
    output = []
    for token in tokens:
        output.extend(token.replace("-", " ").split())
    return output

# normalize_unicode_fractions is lossless, so it's also applied to steps_tokens (the seq2seq target).
# The other transforms (dropping quantities/measurements, artifacts, splitting hyphenated words)
# change the actual content/surface form, so they stay BoW-only in steps_tokens_bow.
df_clean_no_outliers["steps_tokens"] = df_clean_no_outliers["steps_tokens"].apply(normalize_unicode_fractions)

df_clean_no_outliers["steps_tokens_bow"] = df_clean_no_outliers["steps_tokens"].apply(remove_numbers_measurements)
df_clean_no_outliers["steps_tokens_bow"] = df_clean_no_outliers["steps_tokens_bow"].apply(remove_artifacts)
df_clean_no_outliers["steps_tokens_bow"] = df_clean_no_outliers["steps_tokens_bow"].apply(split_hyphenated)

Branch: BoW features for the content classifier (steps_tokens_bow)

steps_tokens stays untouched as the seq2seq target. steps_tokens_bow is a copy with stopwords/punctuation removed, used only for the BoW/multi-task classifier.

In [13]:
df_clean_no_outliers["steps_tokens_bow"] = df_clean_no_outliers["steps_tokens_bow"].apply(remove_stopwords)
df_clean_no_outliers["steps_tokens_bow"] = df_clean_no_outliers["steps_tokens_bow"].apply(remove_punctuation)

df_clean_no_outliers.to_csv('./data/recipes_final.csv')

Train/val/test split

In [14]:
from sklearn import model_selection

df = pd.read_csv('./data/recipes_final.csv', index_col=0)
df['steps_tokens'] = df['steps_tokens'].apply(ast.literal_eval)
df['steps_tokens_bow'] = df['steps_tokens_bow'].apply(ast.literal_eval)

X = df[['steps_tokens', 'steps_tokens_bow']]
y = df['cuisine']

X_train, X_temp, y_train, y_temp = model_selection.train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = model_selection.train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

In [15]:
train_df = X_train.join(y_train)
val_df = X_val.join(y_val)
test_df = X_test.join(y_test)

train_df.to_csv('./data/recipes_train.csv')
val_df.to_csv('./data/recipes_val.csv')
test_df.to_csv('./data/recipes_test.csv')

print(train_df.shape, val_df.shape, test_df.shape)

(8637, 3) (1851, 3) (1851, 3)


Word2Vec

In [16]:
from gensim.models import Word2Vec

WORD2VEC_PATH = './data/models/word2vec.wordvectors'

# keep existing vectors on disk if present - retraining would give the seq2seq model
# new random word vectors, making any saved checkpoint (trained on the old vectors)
# incompatible even though its weight shapes still match
if not os.path.exists(WORD2VEC_PATH):
    w2v_model = Word2Vec(
        sentences=train_df['steps_tokens'],
        vector_size=150,
        window=5,
        min_count=2,
        workers=4,
        epochs=15
    )
    w2v_model.wv.save(WORD2VEC_PATH)
else:
    print(f'{WORD2VEC_PATH} already exists, skipping retraining')

./data/models/word2vec.wordvectors already exists, skipping retraining


BoW

In [17]:
from sklearn.feature_extraction.text import CountVectorizer
import joblib

def identity_analyzer(tokens):
    return tokens

BOW_VECTORIZER_PATH = './data/models/bow_vectorizer.joblib'

# keep the existing vectorizer if present - refitting can shift the BoW vocabulary
# indices, which would make the saved checkpoint's content_classifier (trained against
# the old BoW vocabulary) inconsistent
if os.path.exists(BOW_VECTORIZER_PATH):
    vectorizer = joblib.load(BOW_VECTORIZER_PATH)
    X_train_bow = vectorizer.transform(train_df['steps_tokens_bow'])
    print(f'{BOW_VECTORIZER_PATH} already exists, reusing it')
else:
    vectorizer = CountVectorizer(analyzer=identity_analyzer)
    X_train_bow = vectorizer.fit_transform(train_df['steps_tokens_bow'])
    joblib.dump(vectorizer, BOW_VECTORIZER_PATH)

./data/models/bow_vectorizer.joblib already exists, reusing it


In [18]:
import numpy as np
from gensim.models import KeyedVectors

wv = KeyedVectors.load('./data/models/word2vec.wordvectors')

seq_len = int(train_df['steps_tokens'].apply(len).quantile(0.95))

def tokens_to_matrix(tokens, wv, seq_len):
    matrix = np.zeros((seq_len, wv.vector_size), dtype=np.float32)
    for i, token in enumerate(tokens[:seq_len]):
        if token in wv:
            matrix[i] = wv[token]
    return matrix

Vocabulary and target indices (for CrossEntropyLoss)

In [19]:
PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN = '<PAD>', '<UNK>', '<SOS>', '<EOS>'

word2idx = dict(wv.key_to_index)
word2idx[PAD_TOKEN] = len(word2idx)
PAD_IDX = word2idx[PAD_TOKEN]
word2idx[UNK_TOKEN] = len(word2idx)
UNK_IDX = word2idx[UNK_TOKEN]
word2idx[SOS_TOKEN] = len(word2idx)
SOS_IDX = word2idx[SOS_TOKEN]
word2idx[EOS_TOKEN] = len(word2idx)
EOS_IDX = word2idx[EOS_TOKEN]

VOCAB_SIZE = len(word2idx)

def tokens_to_indices(tokens, word2idx, seq_len):
    # leave room for EOS, then pad - EOS is included in the loss (unlike PAD),
    # giving the model a signal for where to stop generating
    tokens = tokens[:seq_len - 1]
    indices = [word2idx.get(t, UNK_IDX) for t in tokens]
    indices.append(EOS_IDX)
    indices += [PAD_IDX] * (seq_len - len(indices))
    return indices

GRU Encoder

In [20]:
import torch
import torch.nn as nn

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)

class GRUEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, style_dim):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.content_dim = hidden_dim - style_dim
        self.style_head = nn.Linear(hidden_dim, style_dim)
        self.content_head = nn.Linear(hidden_dim, self.content_dim)

    def forward(self, x):
        _, hidden = self.gru(x)
        hidden = hidden.squeeze(0)                    # (batch, hidden_dim)
        style_latent = self.style_head(hidden)          # (batch, style_dim)
        content_latent = self.content_head(hidden)       # (batch, content_dim)
        return style_latent, content_latent

# encoder turns each recipe into one hidden vector, split via two heads into
# style_latent (e.g. cuisine) and content_latent (recipe content)
input_dim = wv.vector_size
hidden_dim = 512
STYLE_DIM = 64
encoder = GRUEncoder(input_dim, hidden_dim, STYLE_DIM).to(device)

mps


GRU Decoder

In [21]:
vocab_size = VOCAB_SIZE
MAX_LEN = seq_len

class GRUDecoder(nn.Module):
    def __init__(self, hidden_dim, vocab_size, embed_dim, sos_idx, pad_idx, max_len):
        super().__init__()
        self.max_len = max_len
        self.sos_idx = sos_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.gru_cell = nn.GRUCell(embed_dim, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, hidden, target=None, teacher_forcing_ratio=0.5):
        # hidden: (batch, hidden_dim) - encoder context vector, used as initial GRU state
        batch_size = hidden.size(0)
        device = hidden.device
        max_len = target.size(1) if target is not None else self.max_len

        input_idx = torch.full((batch_size,), self.sos_idx, dtype=torch.long, device=device)
        h = hidden
        outputs = []

        for t in range(max_len):
            embedded = self.embedding(input_idx)   # (batch, embed_dim)
            h = self.gru_cell(embedded, h)          # (batch, hidden_dim)
            logits_t = self.fc_out(h)               # (batch, vocab_size)
            outputs.append(logits_t.unsqueeze(1))

            use_teacher_forcing = target is not None and torch.rand(1).item() < teacher_forcing_ratio
            input_idx = target[:, t] if use_teacher_forcing else logits_t.argmax(dim=-1)

        return torch.cat(outputs, dim=1)  # (batch, max_len, vocab_size)

decoder = GRUDecoder(hidden_dim, vocab_size, input_dim, SOS_IDX, PAD_IDX, MAX_LEN).to(device)

Seq2Seq Autoencoder — training (input recipe = target recipe)

In [22]:
from torch.utils.data import Dataset, DataLoader

CUISINE2IDX = {'Italian': 0, 'Indian': 1}
BOW_VOCAB_SIZE = X_train_bow.shape[1]

class RecipeDataset(Dataset):
    def __init__(self, df, wv, word2idx, seq_len, cuisine2idx, bow_matrix):
        self.tokens = df['steps_tokens'].tolist()
        self.cuisines = df['cuisine'].tolist()
        self.wv = wv
        self.word2idx = word2idx
        self.seq_len = seq_len
        self.cuisine2idx = cuisine2idx
        self.bow_matrix = bow_matrix

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, idx):
        tokens = self.tokens[idx]
        x = tokens_to_matrix(tokens, self.wv, self.seq_len)
        y = tokens_to_indices(tokens, self.word2idx, self.seq_len)
        style_label = self.cuisine2idx[self.cuisines[idx]]
        bow_target = (self.bow_matrix[idx].toarray().ravel() > 0).astype(np.float32)  # multi-label word presence
        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y, dtype=torch.long),
            torch.tensor(style_label, dtype=torch.long),
            torch.tensor(bow_target, dtype=torch.float32),
        )

BATCH_SIZE = 32

train_dataset = RecipeDataset(train_df, wv, word2idx, seq_len, CUISINE2IDX, X_train_bow)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

class Seq2SeqAutoencoder(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, x, target=None, teacher_forcing_ratio=0.5, return_latents=False):
        style_latent, content_latent = self.encoder(x)
        context = torch.cat([style_latent, content_latent], dim=-1)
        logits = self.decoder(context, target=target, teacher_forcing_ratio=teacher_forcing_ratio)
        if return_latents:
            return logits, style_latent, content_latent
        return logits

recon_criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
style_criterion = nn.CrossEntropyLoss()
content_criterion = nn.BCEWithLogitsLoss()  # multi-label: each word independently yes/no
adv_criterion = nn.CrossEntropyLoss()

CHECKPOINT_PATH = './data/models/checkpoint.pt'
BEST_RECON_CHECKPOINT_PATH = './data/models/checkpoint_best_recon.pt'

# only build the model/classifiers/optimizers the first time this cell runs
if 'model' not in globals():
    model = Seq2SeqAutoencoder(encoder, decoder).to(device)
    style_classifier = nn.Linear(STYLE_DIM, len(CUISINE2IDX)).to(device)

    # content classifier: content_vector -> predicted BoW word presence
    content_classifier = nn.Sequential(
        nn.Linear(encoder.content_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, BOW_VOCAB_SIZE),
    ).to(device)

    # adversarial style classifier (J_adv(c)): content_latent should carry no style info, so the
    # encoder is trained to maximize this classifier's prediction entropy
    adv_style_classifier = nn.Sequential(
        nn.Linear(encoder.content_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, len(CUISINE2IDX)),
    ).to(device)

    optimizer = torch.optim.Adam(
        list(model.parameters()) + list(style_classifier.parameters()) + list(content_classifier.parameters()),
        lr=1e-3,
    )
    adv_optimizer = torch.optim.Adam(adv_style_classifier.parameters(), lr=1e-3)

    # resume from the best checkpoint on disk (e.g. after a kernel restart or an
    # interrupted run) instead of starting from random init. The checkpoint file only
    # holds weights, not the recon loss it was saved at, so recompute it with one
    # no-grad pass before training resumes - otherwise best_recon_loss would start at
    # inf and the first (likely worse) epoch would overwrite this checkpoint.
    if os.path.exists(BEST_RECON_CHECKPOINT_PATH):
        model.load_state_dict(torch.load(BEST_RECON_CHECKPOINT_PATH, map_location=device))
        model.eval()
        total_recon_loss = 0.0
        with torch.no_grad():
            for batch_x, batch_y, _, _ in train_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                logits = model(batch_x, target=batch_y, teacher_forcing_ratio=0.0)
                total_recon_loss += recon_criterion(logits.permute(0, 2, 1), batch_y).item()
        best_recon_loss = total_recon_loss / len(train_loader)
        print(f'Resumed from {BEST_RECON_CHECKPOINT_PATH}, recon loss {best_recon_loss:.4f}')
    else:
        print(f'No checkpoint found at {BEST_RECON_CHECKPOINT_PATH}, training from scratch')

NUM_EPOCHS = 5
TEACHER_FORCING_RATIO = 0.5
STYLE_LOSS_WEIGHT = 1.0
CONTENT_LOSS_WEIGHT = 1.0
ADV_LOSS_WEIGHT = 0.5

# track the best reconstruction loss seen so far (across reruns of this cell too) and
# checkpoint the model whenever it improves - recon_loss doesn't monotonically improve
# (style/adv losses can pull it back up), so the final epoch isn't necessarily the best.
best_recon_loss = globals().get('best_recon_loss', float('inf'))

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    style_classifier.train()
    content_classifier.train()
    adv_style_classifier.train()
    total_loss = 0.0
    total_recon_loss = 0.0
    total_style_loss = 0.0
    total_content_loss = 0.0
    total_adv_encoder_loss = 0.0
    total_adv_clf_loss = 0.0
    correct_adv = 0
    total_adv = 0

    for batch_x, batch_y, batch_style, batch_bow in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        batch_style = batch_style.to(device)
        batch_bow = batch_bow.to(device)
        optimizer.zero_grad()

        logits, style_latent, content_latent = model(
            batch_x, target=batch_y, teacher_forcing_ratio=TEACHER_FORCING_RATIO, return_latents=True
        )
        recon_loss = recon_criterion(logits.permute(0, 2, 1), batch_y)   # (batch, vocab_size, seq_len) vs (batch, seq_len)

        style_logits = style_classifier(style_latent)
        style_loss = style_criterion(style_logits, batch_style)

        content_logits = content_classifier(content_latent)
        content_loss = content_criterion(content_logits, batch_bow)

        adv_logits_for_encoder = adv_style_classifier(content_latent)
        adv_log_probs = torch.log_softmax(adv_logits_for_encoder, dim=-1)
        adv_entropy = -(adv_log_probs.exp() * adv_log_probs).sum(dim=-1).mean()
        adv_encoder_loss = -adv_entropy

        loss = (
            recon_loss
            + STYLE_LOSS_WEIGHT * style_loss
            + CONTENT_LOSS_WEIGHT * content_loss
            + ADV_LOSS_WEIGHT * adv_encoder_loss
        )
        loss.backward()
        optimizer.step()

        adv_optimizer.zero_grad()
        adv_clf_logits = adv_style_classifier(content_latent.detach())
        adv_clf_loss = adv_criterion(adv_clf_logits, batch_style)
        adv_clf_loss.backward()
        adv_optimizer.step()

        total_loss += loss.item()
        total_recon_loss += recon_loss.item()
        total_style_loss += style_loss.item()
        total_content_loss += content_loss.item()
        total_adv_encoder_loss += adv_encoder_loss.item()
        total_adv_clf_loss += adv_clf_loss.item()
        correct_adv += (adv_clf_logits.argmax(dim=-1) == batch_style).sum().item()
        total_adv += batch_style.size(0)

    n_batches = len(train_loader)
    avg_recon_loss = total_recon_loss / n_batches
    print(
        f'epoch {epoch}/{NUM_EPOCHS} '
        f'| loss {total_loss / n_batches:.4f} '
        f'| recon {avg_recon_loss:.4f} | style {total_style_loss / n_batches:.4f} '
        f'| content {total_content_loss / n_batches:.4f} '
        f'| adv_enc {total_adv_encoder_loss / n_batches:.4f} | adv_clf {total_adv_clf_loss / n_batches:.4f} '
        f'| adv_clf_acc {correct_adv / total_adv:.4f}'
    )

    if avg_recon_loss < best_recon_loss:
        best_recon_loss = avg_recon_loss
        torch.save(model.state_dict(), BEST_RECON_CHECKPOINT_PATH)
        print(f'  -> new best recon loss {best_recon_loss:.4f}, saved to {BEST_RECON_CHECKPOINT_PATH}')

torch.save(model.state_dict(), CHECKPOINT_PATH)

# everything downstream (style vectors, style transfer generation) should run on the
# best checkpoint seen so far, not whichever epoch this run happened to end on
model.load_state_dict(torch.load(BEST_RECON_CHECKPOINT_PATH, map_location=device))

Resumed from ./data/models/checkpoint_best_recon.pt, recon loss 5.5353
epoch 1/5 | loss 2.4438 | recon 2.7082 | style 0.0214 | content 0.0318 | adv_enc -0.6352 | adv_clf 0.6706 | adv_clf_acc 0.6640
  -> new best recon loss 2.7082, saved to ./data/models/checkpoint_best_recon.pt
epoch 2/5 | loss 2.5471 | recon 2.7450 | style 0.0061 | content 0.0153 | adv_enc -0.4384 | adv_clf 1.0069 | adv_clf_acc 0.6715
epoch 3/5 | loss 2.4182 | recon 2.6894 | style 0.0024 | content 0.0141 | adv_enc -0.5754 | adv_clf 0.8274 | adv_clf_acc 0.6699
  -> new best recon loss 2.6894, saved to ./data/models/checkpoint_best_recon.pt
epoch 4/5 | loss 2.3732 | recon 2.6587 | style 0.0010 | content 0.0133 | adv_enc -0.5997 | adv_clf 0.7581 | adv_clf_acc 0.6642
  -> new best recon loss 2.6587, saved to ./data/models/checkpoint_best_recon.pt
epoch 5/5 | loss 2.3447 | recon 2.6487 | style 0.0005 | content 0.0128 | adv_enc -0.6346 | adv_clf 0.6947 | adv_clf_acc 0.6558
  -> new best recon loss 2.6487, saved to ./data/mo

<All keys matched successfully>

In [23]:
idx2word = {idx: word for word, idx in word2idx.items()}

model.eval()
with torch.no_grad():
    sample_x, sample_y, sample_style, sample_bow = next(iter(train_loader))
    sample_x = sample_x.to(device)
    logits = model(sample_x, target=None, teacher_forcing_ratio=0.0)  # free-running generation, no teacher forcing
    predicted_indices = logits.argmax(dim=-1)  # (batch, seq_len)

def indices_to_tokens(indices):
    # stop at the first EOS - anything after it is not part of the generated content
    tokens = []
    for i in indices:
        if i == EOS_IDX:
            break
        if i != PAD_IDX:
            tokens.append(idx2word[i])
    return tokens

recipe_idx = 0
original_tokens = indices_to_tokens(sample_y[recipe_idx].tolist())
predicted_tokens = indices_to_tokens(predicted_indices[recipe_idx].tolist())

print('ORIGINAL: ', ' '.join(original_tokens))
print('PREDICTED:', ' '.join(predicted_tokens))

ORIGINAL:  combine bread crumbs with parmesan parsley chives tarragon and pepper mix onion seasoned bread crumbs and melted butter cut tomatoes in half horizontally and seed them fill with bread crumbs and bake in buttered baking dish for about ten minutes at 350 degrees starting with room temperature tomatoes reduces cooking time and helps assure a firm tomato is served to guests
PREDICTED: combine bread crumbs with parmesan parsley chives tarragon and and mix gently shape into mixture into 4 loaves horizontally seed seed and place in bread crumbs bake in buttered baking dish spoon about ten minutes or until cheese is melted and way through preheat oven to 350°f degrees slice loaf into bread crumbs cheese guests to spoon to baking <UNK> guests to spoon to baking <UNK> guests to spoon to baking to guests to spoon to guests to spoon to baking baking dish and gently partially served with heated through


Enc(S_T) — average style vector per cuisine class

In [24]:
idx2cuisine = {idx: cuisine for cuisine, idx in CUISINE2IDX.items()}

model.eval()
style_latent_sums = {cuisine: torch.zeros(STYLE_DIM) for cuisine in CUISINE2IDX}
style_latent_counts = {cuisine: 0 for cuisine in CUISINE2IDX}

with torch.no_grad():
    for batch_x, batch_y, batch_style, batch_bow in train_loader:
        batch_x = batch_x.to(device)
        style_latent, _ = model.encoder(batch_x)
        style_latent = style_latent.cpu()
        for latent, style_idx in zip(style_latent, batch_style):
            cuisine = idx2cuisine[style_idx.item()]
            style_latent_sums[cuisine] += latent
            style_latent_counts[cuisine] += 1

enc_style_by_cuisine = {
    cuisine: style_latent_sums[cuisine] / style_latent_counts[cuisine]
    for cuisine in CUISINE2IDX
}

Style transfer: swap style vector and decode

In [25]:
torch.save(enc_style_by_cuisine['Indian'], './data/models/style_avg_indian.pt')
torch.save(enc_style_by_cuisine['Italian'], './data/models/style_avg_italian.pt')

In [26]:
N_PER_CUISINE = 6
test_samples = pd.concat([
    g.sample(n=min(N_PER_CUISINE, len(g)), random_state=42)
    for _, g in val_df.groupby('cuisine')
]).join(df[['name', 'steps']])
print(test_samples['cuisine'].value_counts())
print(test_samples['steps_tokens'].apply(len).describe())

cuisine
Indian     6
Italian    6
Name: count, dtype: int64
count     12.000000
mean     115.583333
std       48.304824
min       34.000000
25%       75.750000
50%      125.500000
75%      145.000000
max      202.000000
Name: steps_tokens, dtype: float64


In [27]:
print(f'style_dim={STYLE_DIM}, content_dim={encoder.content_dim}')

model.eval()

def encode_recipe(tokens):
    x = tokens_to_matrix(tokens, wv, seq_len)
    x = torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        style_latent, content_latent = model.encoder(x)
    return style_latent.squeeze(0).cpu(), content_latent.squeeze(0).cpu()

def decode_latent(latent, no_repeat_ngram_size=3):
    # greedy decoding - always takes the highest-probability next token (argmax).
    # no_repeat_ngram_size blocks any token that would recreate an n-gram already generated
    hidden = latent.unsqueeze(0).to(device)
    input_idx = torch.full((1,), model.decoder.sos_idx, dtype=torch.long, device=device)
    h = hidden
    generated = []
    seen_ngram_prefixes = {}  # (n-1)-gram prefix -> set of tokens seen following it
    with torch.no_grad():
        for _ in range(model.decoder.max_len):
            embedded = model.decoder.embedding(input_idx)
            h = model.decoder.gru_cell(embedded, h)
            logits_t = model.decoder.fc_out(h)

            # never emit <UNK> - it's a training-target artifact for rare words, not a
            # real word we want to see in generated output, so it's excluded at decode time
            logits_t[0, UNK_IDX] = float('-inf')

            if no_repeat_ngram_size and len(generated) >= no_repeat_ngram_size - 1:
                prefix = tuple(generated[-(no_repeat_ngram_size - 1):])
                banned = seen_ngram_prefixes.get(prefix)
                if banned:
                    logits_t[0, list(banned)] = float('-inf')

            input_idx = logits_t.argmax(dim=-1)

            token = input_idx.item()
            if no_repeat_ngram_size and len(generated) >= no_repeat_ngram_size - 1:
                prefix = tuple(generated[-(no_repeat_ngram_size - 1):])
                seen_ngram_prefixes.setdefault(prefix, set()).add(token)
            generated.append(token)
    return ' '.join(indices_to_tokens(generated))

target_cuisines = ['Italian' if c == 'Indian' else 'Indian' for c in test_samples['cuisine']]
generated_texts_greedy = []

for tokens, target_cuisine in zip(test_samples['steps_tokens'], target_cuisines):
    _, content_latent = encode_recipe(tokens)
    transferred_latent = torch.cat([enc_style_by_cuisine[target_cuisine], content_latent], dim=-1)
    generated_texts_greedy.append(decode_latent(transferred_latent))

style_dim=64, content_dim=448


In [28]:
results = pd.DataFrame({
    'recipe_name': test_samples['name'].tolist(),
    'original_cuisine': test_samples['cuisine'].tolist(),
    'original_text': test_samples['steps'].tolist(),
    'generated_text_greedy': generated_texts_greedy,
    'target_cuisine': target_cuisines,
})
results.to_csv('./data/style_transfer_results.csv')
results

,recipe_name,original_cuisine,original_text,generated_text_greedy,target_cuisine
0,roasted red bell pepper butter,Indian,"cut bell pepper in half. grill, turning oregul...",to make the sauce in a large saucepan heat the...,Italian
1,nita mehta's shahi kaaju aloo,Indian,wash potatoes and peel. cut potatoes into 1 in...,spread the whole ones on the side of the knife...,Italian
2,bhindi bhaji,Indian,wash bhindi in water and dry it using cloth or...,dust first 10 ingredients in a shallow dish an...,Italian
3,ultimate lamb curry - tyler florence,Indian,start heating vegetable oil in a heavy dutch o...,while heating the cooking heat the oil in a la...,Italian
4,vegetable idli,Indian,heat oil in a pan. add mustard seeds. allow to...,in a sauce pan add olive oil and garlic and an...,Italian
5,indian scrambled eggs,Indian,melt butter. add chilies and onions. cook unti...,heat butter and sauce in a add garlic seasonin...,Italian
6,george's pizza dough,Italian,"heat water to 105 to 115°f, dissolve sugar in ...",bring the water to a boil in a large pot add t...,Indian
7,"onion confit, walnut and gorgonzola pizza",Italian,heat the oil in a large saute pan. add the oni...,add the flour in a large pan and add the the t...,Indian
8,classic minestrone,Italian,"in a soup pot, heat olive oil over medium flam...",add the lentils in a large saucepan over mediu...,Indian
9,panna cotta (coffee/vanilla flavoured),Italian,"first, soak the gelatine leaves in a small bow...",pour the first 8 ingredients in a pan and add ...,Indian


Held-out test set evaluation

In [29]:
from torch.utils.data import TensorDataset

def build_eval_tensors(df, wv, word2idx, seq_len, cuisine2idx):
    xs = [tokens_to_matrix(t, wv, seq_len) for t in df['steps_tokens']]
    ys = [tokens_to_indices(t, word2idx, seq_len) for t in df['steps_tokens']]
    styles = [cuisine2idx[c] for c in df['cuisine']]
    return (
        torch.tensor(np.array(xs), dtype=torch.float32),
        torch.tensor(ys, dtype=torch.long),
        torch.tensor(styles, dtype=torch.long),
    )

test_x, test_y, test_style = build_eval_tensors(test_df, wv, word2idx, seq_len, CUISINE2IDX)
test_loader = DataLoader(TensorDataset(test_x, test_y, test_style), batch_size=BATCH_SIZE)

model.eval()
style_classifier.eval()
adv_style_classifier.eval()
total_recon, correct_style, correct_adv, n_batches, n_rows = 0.0, 0, 0, 0, 0
with torch.no_grad():
    for batch_x, batch_y, batch_style in test_loader:
        batch_x, batch_y, batch_style = batch_x.to(device), batch_y.to(device), batch_style.to(device)
        logits, style_latent, content_latent = model(
            batch_x, target=batch_y, teacher_forcing_ratio=0.0, return_latents=True
        )
        total_recon += recon_criterion(logits.permute(0, 2, 1), batch_y).item()
        correct_style += (style_classifier(style_latent).argmax(dim=-1) == batch_style).sum().item()
        correct_adv += (adv_style_classifier(content_latent).argmax(dim=-1) == batch_style).sum().item()
        n_batches += 1
        n_rows += batch_style.size(0)

print(f'test recon loss: {total_recon / n_batches:.4f}')
print(f'test style_clf accuracy (style latent -> cuisine): {correct_style / n_rows:.4f}')
print(f'test adv_clf accuracy (content latent -> cuisine, chance = 0.5): {correct_adv / n_rows:.4f}')

test recon loss: 6.8353
test style_clf accuracy (style latent -> cuisine): 0.9335
test adv_clf accuracy (content latent -> cuisine, chance = 0.5): 0.5370
